# EviPattern — Phi-3.5-mini generation on Colab T4

Runs only `02b_generate_and_label.py` on GPU. Downstream (02c, 04b, 05b, 06b) runs locally on Windows in <5 min.

**Before running:** Runtime → Change runtime type → T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1 — install dependencies (~30 seconds)
!pip install -q transformers==4.46.3 accelerate sentence-transformers tqdm pyyaml

In [ ]:
# Cell 2 — verify GPU
import torch
assert torch.cuda.is_available(), "GPU not enabled. Runtime → Change runtime type → T4 GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — mount Drive and copy your zip into Colab's workspace
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS LINE: path to the zip you uploaded to Drive
ZIP_IN_DRIVE = '/content/drive/MyDrive/evistate_phi.zip'

!cp "$ZIP_IN_DRIVE" /content/evistate_phi.zip
!unzip -q /content/evistate_phi.zip -d /content/evistate
%cd /content/evistate
!ls -la

In [ ]:
# Cell 3b — fix the flat zip layout
import shutil, os
base = '/content/evistate'
for sub in ['scripts', 'config', 'outputs_phi']:
    os.makedirs(f'{base}/{sub}', exist_ok=True)
shutil.move(f'{base}/02b_generate_and_label.py', f'{base}/scripts/02b_generate_and_label.py')
shutil.move(f'{base}/phi.yaml',                  f'{base}/config/phi.yaml')
shutil.move(f'{base}/retrievals.pkl',            f'{base}/outputs_phi/retrievals.pkl')
!ls -la {base}/scripts {base}/config {base}/outputs_phi

In [ ]:
# Cell 4 — run the actual 02b script.
!python scripts/02b_generate_and_label.py --config config/phi.yaml

In [ ]:
# Cell 5 — sanity-check the output (should print 800)
!wc -l outputs_phi/generations.jsonl

In [ ]:
# Cell 6 — copy generations.jsonl back to your Drive
!cp outputs_phi/generations.jsonl /content/drive/MyDrive/phi_generations.jsonl
print('Done. Download phi_generations.jsonl from your Drive.')